In [1]:
import carla, random, csv, os, time
import numpy as np
import cv2

In [2]:
client = carla.Client('localhost', 2000)
client.set_timeout(10.0)
world = client.get_world()
bp_lib = world.get_blueprint_library()

client.load_world('Town04')

In [3]:
base_dir = r'D:\CARLA data\data'
run_dir = base_dir
run_number = 0
while os.path.exists(run_dir):
    run_number += 1
    run_dir = f'{base_dir}_{run_number}'

images_dir = os.path.join(run_dir, 'images')
os.makedirs(images_dir)
log_path = os.path.join(run_dir, 'log.csv')
log_file = open(log_path, 'x', newline='')

writer = csv.writer(log_file)
writer.writerow(['image', 'steer'])
print(f'Recording data will be saved to: {run_dir}')

Recording data will be saved to: D:\CARLA data\data_1


In [4]:
spectator = world.get_spectator()
transform = spectator.get_transform()
location = transform.location
rotation = transform.rotation
transform = spectator.get_transform()

# vehicle = world.spawn_actor(bp_lib.find('vehicle.ford.mustang'), random.choice(world.get_map().get_spawn_points()))
vehicle = world.spawn_actor(bp_lib.find('vehicle.ford.mustang'), transform)

cam_bp = bp_lib.find('sensor.camera.rgb')
cam_bp.set_attribute('image_size_x', '1200')
cam_bp.set_attribute('image_size_y', '1000')
camera = world.spawn_actor(cam_bp, carla.Transform(carla.Location(x=1.5, z=1.5)), attach_to=vehicle)

In [5]:
traffic_manager = client.get_trafficmanager()
traffic_manager.set_synchronous_mode(False)
traffic_manager.set_global_distance_to_leading_vehicle(5.0)
traffic_manager.auto_lane_change(vehicle, False)

waypoint = world.get_map().get_waypoint(
    vehicle.get_location(),
    project_to_road=True,
    lane_type=carla.LaneType.Driving
)

ring_route = []

for _ in range(1500):
    ring_route.append(waypoint.transform.location)

    next_waypoints = waypoint.next(2.0)
    if not next_waypoints:
        break

    forward = waypoint.transform.get_forward_vector()

    waypoint = max(
        next_waypoints,
        key=lambda candidate: (
            forward.x * candidate.transform.get_forward_vector().x
            + forward.y * candidate.transform.get_forward_vector().y
        )
    )

vehicle.set_autopilot(True, traffic_manager.get_port())
traffic_manager.set_path(vehicle, ring_route)

print(f"Using fixed route with {len(ring_route)} waypoints")

Using fixed route with 1500 waypoints


In [6]:
def on_image(image):
    if not recording_active:
        return

    if image.frame % 5 != 0:
        return

    frame = np.frombuffer(image.raw_data, dtype=np.uint8)
    frame = frame.reshape((image.height, image.width, 4))[:, :, :3]

    name = f'{image.frame:06d}.png'
    cv2.imwrite(os.path.join(images_dir, name), frame)

    steer = vehicle.get_control().steer
    writer.writerow([name, steer])

In [7]:
recording_active = True
camera.listen(on_image)
print('Recording started')

Recording started


In [8]:
RECORD_SECONDS = 1000
print(f'Recording for {RECORD_SECONDS} seconds...')
time.sleep(RECORD_SECONDS)
print('Done recording')

Recording for 1000 seconds...
Done recording


In [9]:
camera.stop()
recording_active = False
time.sleep(1)

camera.destroy()
vehicle.destroy()

log_file.flush()
log_file.close()
print('Cleaned up')

Cleaned up
